In [1]:
import os
import sys
import random
import pandas as pd
from datasets import Dataset

# Add parent directory to path to access utility modules
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Import the existing functions from utils
from utils.data import get_cl_learning_data, prepare_datasets_qa

# Load all datasets using the existing function
print("Loading all datasets...")
dataset_dict = get_cl_learning_data()

# Display information about each dataset
for dataset_name, dataset_samples in dataset_dict.items():
    print(f"\n==== {dataset_name} Dataset ====")
    print(f"Total samples: {len(dataset_samples)}")
    
    # Sample 3 random examples and format them using prepare_datasets_qa
    print(f"\nSample prompts from {dataset_name}:")
    sample_indices = random.sample(range(len(dataset_samples)), min(3, len(dataset_samples)))
    
    for idx in sample_indices:
        # Get original question and answer
        sample = dataset_samples[idx]
        print(f"\nOriginal format:")
        print(f"Question: {sample['question']}")
        print(f"Answer: {sample['answer']}")
        
        # Format using the prepare_datasets_qa function
        formatted = prepare_datasets_qa(sample)
        print(f"\nFormatted prompt:")
        print(formatted['prompt'])
        print("-" * 50)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading all datasets...
Loading ASDiv dataset...
Loaded 2305 problems from ASDiv
Formatting ASDiv samples...
Added 2305 formatted examples from ASDiv
Loading ParaMAWPS dataset...
Loaded 13023 problems from ParaMAWPS
Formatting ParaMAWPS samples...
Added 13023 formatted examples from ParaMAWPS
Loading DMath dataset...
Loaded 7943 problems from DMath
Formatting DMath samples...
Added 7943 formatted examples from DMath
Total examples across all datasets: 23271

==== ASDiv Dataset ====
Total samples: 2305

Sample prompts from ASDiv:

Original format:
Question: For finishing touches, he needed 70 gallons of paint. If he bought 23 gallons to add to his existing 36 gallons of paint, how much more paint will he need?
Answer: The answer is 11 (gallons).

Formatted prompt:
###Question: For finishing touches, he needed 70 gallons of paint. If he bought 23 gallons to add to his existing 36 gallons of paint, how much more paint will he need?
###Answer: The answer is 11 (gallons).
------------------

In [1]:
import os
import sys
import json
import xml.etree.ElementTree as ET
import random

def prepare_datasets_qa(example):
    # Simple Question-Answer format
    example['prompt'] = f"###Question: {example['question']}\n###Answer: {example['answer']}"
    return example

# Data loading functions for each dataset format
def load_asdiv_data(file_path):
    """Load and parse ASDiv XML data"""
    tree = ET.parse(file_path)
    root = tree.getroot()
    problems = []
    
    for problem in root.findall(".//Problem"):
        body_elem = problem.find("Body")
        question_elem = problem.find("Question")
        answer_elem = problem.find("Answer")
        formula_elem = problem.find("Formula")
        
        if question_elem is not None and answer_elem is not None:
            body = body_elem.text.strip() if body_elem is not None else ""
            question = question_elem.text.strip()
            answer = answer_elem.text.strip()
            formula = formula_elem.text.strip() if formula_elem is not None else ""
            
            # Format the text
            if body and formula:
                text = f"Question: {body} {question}\nSolution: {formula}\nAnswer: {answer}"
            elif formula:
                text = f"Question: {question}\nSolution: {formula}\nAnswer: {answer}"
            elif body:
                text = f"Question: {body} {question}\nAnswer: {answer}"
            else:
                text = f"Question: {question}\nAnswer: {answer}"
                
            problems.append({"text": text})
    
    print(f"Loaded {len(problems)} problems from ASDiv")
    return problems

def load_paramawps_data(file_path):
    """Load and parse ParaMAWPS JSON data"""
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    problems = []
    for item in data:
        question = item.get("original_text", "").strip()
        answer = str(item.get("ans", "")).strip()
        
        # Just use the original text and answer, no solution
        text = f"Question: {question}\nAnswer: {answer}"
        problems.append({"text": text})
    
    print(f"Loaded {len(problems)} problems from ParaMAWPS")
    return problems

def load_dmath_data(file_path):
    """Load and parse DMath JSON data"""
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    problems = []
    for item_id, item_data in data.items():
        question = item_data.get("question_en", "").strip()
        answer = item_data.get("answer_en", "").strip()
        
        # Only use question_en and answer_en
        text = f"Question: {question}\nAnswer: {answer}"
        problems.append({"text": text})
    
    print(f"Loaded {len(problems)} problems from DMath")
    return problems

def get_cl_learning_data():
    # Define data loading functions directly here to avoid import issues
    
    # Find data root directory
    data_root = None
    for root in [os.getcwd()] + [os.path.abspath(os.path.join(os.getcwd(), *['..'] * i)) for i in range(1, 4)]:
        if os.path.exists(os.path.join(root, "data")):
            data_root = root
            break
    
    if data_root is None:
        raise FileNotFoundError("Could not find data directory")
    
    # Fixed format_with_solution function that handles different text patterns
    def format_with_solution(item):
        text = item['text']
        
        # Check if Solution section exists
        if '\nSolution:' in text:
            # Handle case with Solution section
            parts = text.split('Question: ')[1].split('\nSolution:')
            question = parts[0].strip()
            
            solution_answer_parts = parts[1].split('\nAnswer:')
            solution = solution_answer_parts[0].strip()
            answer = solution_answer_parts[1].strip()
            
            return {
                'question': question,
                'answer': f"Let me solve this step by step.\n{solution}\nTherefore, the answer is {answer}."
            }
        else:
            # Handle case without Solution section
            parts = text.split('Question: ')[1].split('\nAnswer:')
            question = parts[0].strip()
            answer = parts[1].strip()
            
            return {
                'question': question,
                'answer': f"The answer is {answer}."
            }
    
    # Define dataset configurations
    datasets_config = [
        {
            "name": "ASDiv",
            "path": os.path.join(data_root, "data", "curriculum_learning", "1_ASDiv", "ASDiv.xml"),
            "loader": load_asdiv_data,
            "format": format_with_solution
        },
        {
            "name": "ParaMAWPS",
            "path": os.path.join(data_root, "data", "curriculum_learning", "2_ParaMAWPS", "ParaMAWPS_trainset.json"),
            "loader": load_paramawps_data,
            "format": format_with_solution
        },
        {
            "name": "DMath",
            "path": os.path.join(data_root, "data", "curriculum_learning", "4_Dmath", "dmath_train.json"),
            "loader": load_dmath_data,
            "format": format_with_solution
        }
    ]
    
    # Process all datasets
    standardized_datasets = {}
    total_examples = 0
    
    for dataset_config in datasets_config:
        try:
            print(f"Loading {dataset_config['name']} dataset...")
            data = dataset_config["loader"](dataset_config["path"])
            standardized_data = []
            
            print(f"Formatting {dataset_config['name']} samples...")
            for item in data:
                try:
                    formatted_item = dataset_config["format"](item)
                    standardized_data.append(formatted_item)
                except Exception as e:
                    print(f"Error formatting item in {dataset_config['name']}: {str(e)}")
                    continue
            
            standardized_datasets[dataset_config["name"]] = standardized_data
            total_examples += len(standardized_data)
            print(f"Added {len(standardized_data)} formatted examples from {dataset_config['name']}")
            
        except Exception as e:
            print(f"Error processing {dataset_config['name']} dataset: {str(e)}")
            standardized_datasets[dataset_config["name"]] = []
    
    print(f"Total examples across all datasets: {total_examples}")
    return standardized_datasets

# Test function to verify the implementation
def test_get_cl_learning_data():
    dataset_dict = get_cl_learning_data()
    
    # Display information about each dataset
    for dataset_name, dataset_samples in dataset_dict.items():
        print(f"\n==== {dataset_name} Dataset ====")
        print(f"Total samples: {len(dataset_samples)}")
        
        # Sample examples if available
        if dataset_samples:
            print(f"\nSample prompts from {dataset_name}:")
            sample_indices = random.sample(range(len(dataset_samples)), min(3, len(dataset_samples)))
            for idx in sample_indices:
                # Get original question and answer
                sample = dataset_samples[idx]
                print(f"\nOriginal format:")
                print(f"Question: {sample['question']}")
                print(f"Answer: {sample['answer']}")
                
                # Format using the prepare_datasets_qa function
                formatted = prepare_datasets_qa(sample)
                print(f"\nFormatted prompt:")
                print(formatted['prompt'])
                print("-" * 50)

if __name__ == "__main__":
    test_get_cl_learning_data()

Loading ASDiv dataset...
Loaded 2305 problems from ASDiv
Formatting ASDiv samples...
Added 2305 formatted examples from ASDiv
Loading ParaMAWPS dataset...
Loaded 13023 problems from ParaMAWPS
Formatting ParaMAWPS samples...
Added 13023 formatted examples from ParaMAWPS
Loading DMath dataset...
Loaded 7943 problems from DMath
Formatting DMath samples...
Added 7943 formatted examples from DMath
Total examples across all datasets: 23271

==== ASDiv Dataset ====
Total samples: 2305

Sample prompts from ASDiv:

Original format:
Question: At the produce store you can buy 2 bags of bananas for $12.46. How much would it cost if you were to buy 6 bags?
Answer: Let me solve this step by step.
(12.46/2)*6=37.38
Therefore, the answer is 37.38 (dollars).

Formatted prompt:
###Question: At the produce store you can buy 2 bags of bananas for $12.46. How much would it cost if you were to buy 6 bags?
###Answer: Let me solve this step by step.
(12.46/2)*6=37.38
Therefore, the answer is 37.38 (dollars).
